<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi/notebooks/gsm8k/LLaDA_GSM8K_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# LLaDA on GSM8K: inspect one prompt
Start with **one question** from GSM8K's `main` / `test` split and ordinary LLaDA baseline decoding. Follow the same setup, Drive cache, Config Updater, subprocess run, and result-inspection flow as the Sudoku notebooks.

You will see:
1. The question and instructions assembled into the user message.
2. The checkpoint's chat formatting and the resulting input token IDs.
3. The generated response and a separate comparison with the reference answer.

Select a CUDA GPU runtime that passes the check below. The frozen LLaDA model uses BF16 and loads once per inference subprocess. The notebook kernel displays saved files; it never loads the 8B model. This notebook reuses the **same cached LLaDA snapshot** as Sudoku.

For the private repository, authorize Colab to read GitHub or upload this notebook. The clone cell needs a read-only `GITHUB_TOKEN` in Colab Secrets or the hidden prompt. The new notebook and runner must be present on `adi` before using the repository checkout.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select an L4 or A100 GPU runtime, then reconnect.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | GPU memory: {gpu.total_memory / 2**30:.1f} GiB")
if gpu.total_memory < 20 * 2**30:
    raise RuntimeError("This notebook uses unquantized BF16 weights. Choose an L4/A100 runtime with more GPU memory; a 16 GB T4 is too small for this setup.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This GPU does not support the configured BF16 inference.")


## 1. Get the code from `adi`
Clone or update `adi`. A clean checkout on another branch can switch safely; local modifications and divergent commits are never discarded.


In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

REPOSITORY = "CooL-Legend/CSCI544-DLM"
BRANCH = "adi"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
REPO_DIR = Path("/content/CSCI544-DLM")

def run_git(args, token=None):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    with tempfile.TemporaryDirectory() as tmp:
        if token:
            askpass = Path(tmp) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['COLAB_GH_TOKEN'])\n"
            )
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), COLAB_GH_TOKEN=token)
        return subprocess.run(["git", "-c", "credential.helper=", *args], env=env, capture_output=True, text=True)

def remote_git(args):
    result = run_git(args)
    if result.returncode:
        from google.colab import userdata
        try:
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if not token:
            token = getpass.getpass("GitHub token with read access to this private repository: ")
        try:
            result = run_git(args, token)
        finally:
            del token
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or "GitHub clone/update failed.")

def ensure_checkout():
    def local_git(*args):
        return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Choose another REPO_DIR.")
        if local_git("remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError("The existing checkout belongs to a different repository.")
        if local_git("status", "--porcelain"):
            raise RuntimeError("The checkout has local changes. Save them or choose another REPO_DIR before updating.")
        fetch_specs = local_git("config", "--get-all", "remote.origin.fetch").splitlines()
        branch_ref = f"refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
        if not any(spec.lstrip("+") in (branch_ref, "refs/heads/*:refs/remotes/origin/*") for spec in fetch_specs):
            local_git("remote", "set-branches", "--add", "origin", BRANCH)
        remote_git(["-C", str(REPO_DIR), "fetch", "origin", branch_ref])
        exists = subprocess.run(["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"]).returncode == 0
        if exists:
            local_git("switch", BRANCH)
        else:
            local_git("switch", "--create", BRANCH, "--track", f"origin/{BRANCH}")
        local_git("merge", "--ff-only", f"origin/{BRANCH}")
    else:
        remote_git(["clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    if local_git("branch", "--show-current") != BRANCH or local_git("rev-parse", "HEAD") != local_git("rev-parse", f"origin/{BRANCH}"):
        raise RuntimeError("The checkout differs from origin/adi. Review local commits before running experiments.")
    print(f"Code ready: {REPO_DIR} | branch: {BRANCH} | commit: {local_git('rev-parse', '--short', 'HEAD')}")

ensure_checkout()


## 2. Install LLaDA dependencies
Keep Colab's CUDA-enabled PyTorch. Inference always uses a fresh Python process, which sees the installed dependency versions. LLaDA retains Transformers 4.38.2.

If you switch model notebooks in the same runtime, run this installation cell again before inference. The two model profiles use different Transformers versions.


In [ ]:
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(REPO_DIR / "requirements/llada-colab.txt"),
], check=True)

# Install our package without replacing Colab's CUDA-enabled PyTorch.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR)], check=True)
# Refresh editable-package paths in this already-running notebook kernel.
import site
for package_dir in [*site.getsitepackages(), site.getusersitepackages()]:
    site.addsitedir(package_dir)


## 3. Mount Drive
Allow at least **20 GB free Drive storage per model snapshot**, plus dataset cache and experiment outputs. Downloading both models needs room for both copies. Colab Pro compute and Drive storage are separate allowances.

`COPY_MODEL_TO_RUNTIME = True` stages the persistent model on runtime disk. Set it to `False` to load directly from Drive. Existing experiment folders are not moved or deleted.


In [ ]:
from google.colab import drive
import importlib

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/CSCI544-DLM")
MODEL_FAMILY = "llada"
from dlm_project.models import profiles as model_profiles
from dlm_project.experiments import paths as experiment_paths
importlib.reload(model_profiles)
importlib.reload(experiment_paths)
profile = model_profiles.get_profile(MODEL_FAMILY)
MODEL_NAME = profile["repo_id"].split("/")[-1]
DRIVE_MODEL_DIR = DRIVE_ROOT / "models" / MODEL_NAME
RUNTIME_MODEL_DIR = Path("/content/models") / MODEL_NAME
COPY_MODEL_TO_RUNTIME = True
EXPERIMENT_TIMEZONE = "America/Los_Angeles"

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(DRIVE_ROOT / "cache/hub")
os.environ["HF_DATASETS_CACHE"] = str(DRIVE_ROOT / "cache/datasets")
MODEL_PATH = RUNTIME_MODEL_DIR if COPY_MODEL_TO_RUNTIME else DRIVE_MODEL_DIR
print("Persistent model:", DRIVE_MODEL_DIR)
print("Experiment root:", DRIVE_ROOT / profile["label"] / "GSM8K")


## 4. Load the GSM8K configuration
Use the same model/cache locations and timezone as Sudoku. Outputs go under `LLaDA/GSM8K/`, keeping this task's results together. Edit the question and instructions in Config Updater below.


In [ ]:
import yaml

config = yaml.safe_load((REPO_DIR / "configs/gsm8k/llada.yaml").read_text())
config["data"]["cache_dir"] = os.environ["HF_DATASETS_CACHE"]
config["model"].update(family=MODEL_FAMILY, model_path=str(MODEL_PATH), device="cuda", local_files_only=True)
config["experiment_root"] = str(DRIVE_ROOT)
config["experiment_timezone"] = EXPERIMENT_TIMEZONE
RUN_CONFIG = Path("/content/llada-gsm8k-run.yaml")
RUNNER = REPO_DIR / "scripts/run_gsm8k.py"
print("Model revision:", profile["revision"])


## 5. Cache the model on Drive
Download the pinned model's tokenizer, chat template, configuration, Python implementation, and all shards listed in its weight index. Later sessions reuse the same files; keep the model folder's hidden `.cache` directory.

Rerun this cell after an interrupted download. If validation reports missing files or insufficient disk space, resolve that before inference. No weights are committed to GitHub.


In [ ]:
command = [sys.executable, "-u", str(REPO_DIR / "scripts/cache_model.py"), "--model", MODEL_FAMILY, "--drive-dir", str(DRIVE_MODEL_DIR)]
if COPY_MODEL_TO_RUNTIME:
    command += ["--runtime-dir", str(RUNTIME_MODEL_DIR)]
subprocess.run(command, check=True)


## Config Updater
Start with `QUESTION_INDEX = 0` and `CUSTOM_QUESTION = None` to use the first GSM8K test question. Change the index to try another example, or set `CUSTOM_QUESTION` to your own word problem. Custom questions have no reference score.

Edit `PROMPT_INSTRUCTIONS` to see how instructions affect the model's response. Keep the requested final line `#### <number>` if you want automatic answer parsing. The reference solution is **never included in the prompt**.

`gen_length` is the number of **output token positions**, not words or input length. This walkthrough uses baseline generation with 256 output positions, 256 steps, and 128-token blocks. A response can be incomplete at this fixed length; the raw output remains visible.


In [ ]:
NUM_SAMPLES = 1  # This walkthrough intentionally runs one prompt at a time.
QUESTION_INDEX = 0
CUSTOM_QUESTION = None  # Example: "Mia has 3 apples and buys 2 more. How many apples does she have?"
PROMPT_INSTRUCTIONS = (
    "Solve the following math word problem. Explain your solution briefly.\n"
    "End your response with a line in this exact format: #### <number>"
)

config["data"].update(num_samples=NUM_SAMPLES, split="test", question_index=QUESTION_INDEX)
config["prompt"].update(instructions=PROMPT_INSTRUCTIONS, custom_question=CUSTOM_QUESTION)
config["generation"].update(gen_length=256, steps=256, block_length=128,
                            temperature=0.0, remasking="low_confidence")
config["decoding"] = {"method": "baseline", "seed": 1}
config["output_dir"] = str(experiment_paths.experiment_parent(config, dataset_name="GSM8K"))
RUN_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print(RUN_CONFIG.read_text())
print("Experiment output directory:", config["output_dir"])


## Preview the prompt before loading the model
The runner loads one question and builds `instructions + question`. This preview does not load model weights or tokenize text. The JSON chat message shown here contains only the user role and your prompt. The checkpoint's chat template will add its own control tokens during inference.

For GSM8K, the dataset provides separate `question` and `answer` fields. Only `question` enters this preview; `answer` is reserved for the later comparison. [Dataset card](https://huggingface.co/datasets/openai/gsm8k).


In [ ]:
import json
import uuid

PREVIEW_PATH = Path("/content") / f"gsm8k-prompt-{uuid.uuid4().hex}.json"
subprocess.run([
    sys.executable, "-u", str(RUNNER), "--config", str(RUN_CONFIG),
    "--prompt-only", "--preview-path", str(PREVIEW_PATH),
], check=True)
preview = json.loads(PREVIEW_PATH.read_text())
print("Question source:", preview["source"])
print("\nChat message before the HF template:")
print(json.dumps(preview["messages"], indent=2, ensure_ascii=False))


## 6. Run one LLaDA generation
This subprocess loads the cached HF model once on CUDA. The wrapper applies the checkpoint's chat template, tokenizes the user message, appends masked output positions, and calls the existing LLaDA sampler. It does not call autoregressive `model.generate()`.

The runner saves the prompt inspection, raw response, answer comparison, decoding statistics and trace. Each run gets a new timestamp directory. `status: completed` means generation and saving finished; the answer can still be wrong or unparseable.


In [ ]:
latest = None
RUN_MANIFEST = Path("/content") / f"llada-gsm8k-run-{uuid.uuid4().hex}.json"
subprocess.run([
    sys.executable, "-u", str(RUNNER), "--config", str(RUN_CONFIG),
    "--run-manifest", str(RUN_MANIFEST),
], check=True)
receipt = json.loads(RUN_MANIFEST.read_text())
latest = Path(receipt["run_dir"])
print("This run:", latest)


## 7. Inspect the model input and response
The view below comes from the same wrapper formatting/tokenization method used for generation. Special chat tokens are kept visible. The prompt token count excludes the appended output masks. Individual token strings can contain whitespace markers or parts of words.


In [ ]:
import pandas as pd
from IPython.display import display

if latest is None:
    raise RuntimeError("Complete the generation cell before inspecting its results.")
prompt_view = json.loads((latest / "prompt.json").read_text())
print("1. User message:\n" + prompt_view["user_prompt"])
print("\n2. Chat-formatted input decoded with special tokens visible:\n" + prompt_view["rendered_chat_prompt"])
print(f"\n3. Input: {prompt_view['prompt_token_count']} prompt tokens + "
      f"{prompt_view['output_mask_count']} output masks (mask token ID {prompt_view['mask_token_id']})")
TOKEN_ROWS_TO_SHOW = 80
prompt_tokens = pd.DataFrame({
    "position": range(prompt_view["prompt_token_count"]),
    "token_id": prompt_view["input_token_ids"],
    "token": prompt_view["input_tokens"],
})
display(prompt_tokens.head(TOKEN_ROWS_TO_SHOW))
print("Full prompt and token IDs saved in:", latest / "prompt.json")


In [ ]:
record = json.loads((latest / "predictions.jsonl").read_text().strip())
print("4. LLaDA's unchanged response:\n" + record["raw_output"])
print("\n5. Reference solution (evaluation only):")
print(record["reference_solution"] if record["reference_available"] else "No reference for a custom question.")
display(pd.DataFrame([{key: record[key] for key in
                      ["predicted_answer", "reference_answer", "parse_success", "reference_available", "exact_match"]}]))

stats = record["decoding_stats"]
assert torch.device(stats["model_device"]).type == "cuda", "Use this notebook's CUDA configuration."
display(pd.DataFrame([{key: stats[key] for key in
                      ["method", "model_device", "completion_model_calls", "latency_seconds", "peak_gpu_memory_bytes"]}]))
print("Run status:", json.loads((latest / "metrics.json").read_text())["status"])
print("Artifacts:", sorted(path.name for path in latest.iterdir()))


## 8. Try another prompt
Change `PROMPT_INSTRUCTIONS`, `QUESTION_INDEX`, or `CUSTOM_QUESTION` in Config Updater. Rerun that cell, the prompt preview, generation, and the inspection cells. The weights stay fixed; only the input and selected generation settings change. Each invocation reloads the same cached model into its own subprocess, so the kernel does not retain another copy.

For an answer-only experiment, use `PROMPT_INSTRUCTIONS = "Solve the problem. Return only the final answer as #### <number>."` Keep the question fixed to compare the two prompt styles. These are single-example observations, not a benchmark score.

Final-answer parsing uses the last `####` line. It normalizes commas, signs and decimal zeros, and does not guess an answer from other numbers in the response. Missing/malformed answer markers are reported as parse failures. Custom questions have `exact_match = None`, because no reference is available.

Artifacts are stored under `MyDrive/CSCI544-DLM/LLaDA/GSM8K/<generation-config>/<timestamp>/`: `config.json`, `prompt.json`, `response.txt`, `predictions.jsonl`, `metrics.json`, and `trace.jsonl`. Model weights remain in the shared `models/LLaDA-8B-Instruct/` cache.

[HF model](https://huggingface.co/GSAI-ML/LLaDA-8B-Instruct) · [GSM8K dataset](https://huggingface.co/datasets/openai/gsm8k) · [GSM8K source](https://github.com/openai/grade-school-math)
